# ResNet Fixation Generalization Experiment

This notebook trains ImageNet-pretrained ResNet classifiers on the labeled fixation ERP dataset and then applies the trained models to six external ERP/fixation data sources.

The main question is whether the trained model behaves like a singular biased classifier, like a random classifier, or whether it transfers a useful fixation-ERP representation to new data sources.

The ResNet constructors follow the Metalhead API, where `Metalhead.ResNet(depth; pretrain, inchannels, nclasses)` supports ResNet depths including 18 and 34: https://fluxml.ai/Metalhead.jl/dev/api/resnet/

In [1]:
using InteractiveUtils
versioninfo()

Julia Version 1.12.3
Commit 966d0af0fdf (2025-12-15 11:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 16 × AMD Ryzen 7 7800X3D 8-Core Processor
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, znver4)
  GC: Built with stock GC
Threads: 16 default, 1 interactive, 16 GC (on 16 virtual cores)
Environment:
  JULIA_NUM_THREADS = 16


## Experiment Policy

Training uses all currently labeled fixation images. Class instances keep all four modulo-4 parts. No-class instances are also split modulo-4, but only one seeded part is kept to limit class imbalance.

Every image keeps the ERP image convention used in the earlier notebooks: rows are sorted trials on the y-axis and columns are time on the x-axis. The preprocessing pipeline is `sort trials -> z-score -> Gaussian smooth from the shared utils -> resize to 64x64`.

The models run sequentially: first the 18-layer pretrained model, then the 34-layer pretrained model.

In [2]:
import Pkg

# Keep notebook startup stable and reuse the existing Julia environment.
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

const NOTEBOOK_DIR = pwd()
Pkg.activate(joinpath(NOTEBOOK_DIR, "..", "model_test"))

using DataFrames

include(joinpath(NOTEBOOK_DIR, "resnet_fixation_generalization_experiment.jl"))
using .Week20ResNetFixationGeneralization

println("Notebook directory: ", NOTEBOOK_DIR)
println("Output directory: ", Week20ResNetFixationGeneralization.OUTPUT_DIR)

  Activating project at `~/Dokumente/BA2/notebooks/model_test`
  Activating project at `~/Dokumente/BA2/notebooks/model_test`


Notebook directory: /home/benjamin/Dokumente/BA2/notebooks/week_20
Output directory: /home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization


## Target Sources

The target sources are the Week-19 standardized bundles. ERP CORE N2PC and N170 use `reaction_time_ms` with a modulo-2 split. The fixation target sources use `fixation_duration_ms` without a modulo split.

For sources with multiple subjects, channels are merged across subjects following the Week-19 preview convention, so one origin image is one data source, one merged subject group, one channel, and one sort variable. Modulo-split variants keep the same `origin_id`, which allows the consistency analysis to check whether variants from the same origin are classified the same way.

In [3]:
target_specs_df = DataFrame(Week20ResNetFixationGeneralization.TARGET_DATASET_SPECS)
target_specs_df

Row,dataset_key,label,sort_col,mod_split_k,baseline_correct
,String,String,Symbol,Int64,Bool
1,erp_core_n2pc_clean,ERP CORE N2PC,reaction_time_ms,2,true
2,erp_core_n170_clean,ERP CORE N170,reaction_time_ms,2,true
3,eye_eeg_freeviewing_fixations,EYE-EEG FREEVIEWING FIXATIONS,fixation_duration_ms,1,false
4,eye_eeg_reading_fixations,EYE-EEG READING FIXATIONS,fixation_duration_ms,1,false
5,eye_eeg_sceneviewing_tobii_fixations,EYE-EEG SCENEVIEWING TOBII FIXATIONS,fixation_duration_ms,1,false
6,roamm_reading_fixations,ROAMM READING FIXATIONS,fixation_duration_ms,1,false


## Run

This cell performs the full experiment and writes all thesis-ready outputs to `notebooks/week_20/outputs/resnet_fixation_generalization`:

- training metadata and loss history
- per-image target predictions with logits and softmax probabilities
- per-dataset prediction statistics
- mod-split consistency tables for N2PC and N170
- final classifier-head output weights
- confidence example plots for each model and data source

In [4]:
result = Week20ResNetFixationGeneralization.run_experiment();
result.summary_df

CUDA device: NVIDIA GeForce RTX 4070
Output directory: /home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization
Training batch size: 32
Preparing labeled fixation training dataset.
Preparing target datasets.
Materializing target dataset: ERP CORE N2PC
  images: 70 | origins: 35
Materializing target dataset: ERP CORE N170
  images: 70 | origins: 35
Materializing target dataset: EYE-EEG FREEVIEWING FIXATIONS
  images: 25 | origins: 25
Materializing target dataset: EYE-EEG READING FIXATIONS
  images: 66 | origins: 66
Materializing target dataset: EYE-EEG SCENEVIEWING TOBII FIXATIONS
  images: 3 | origins: 3
Materializing target dataset: ROAMM READING FIXATIONS
  images: 64 | origins: 64

Training resnet18_pretrained (1/2)
resnet18_pretrained | epoch 1/8 | loss=0.73745
resnet18_pretrained | epoch 2/8 | loss=0.50128
resnet18_pretrained | epoch 3/8 | loss=0.19743
resnet18_pretrained | epoch 4/8 | loss=0.07248
resnet18_pretrained | epoch 5/8 | loss=0.15265
resnet1

Row,model_name,dataset_key,dataset_label,n_images,n_origins,predicted_class_n,predicted_no_class_n,predicted_class_rate,predicted_no_class_rate,dominant_class,dominant_class_rate,is_single_class,prob_class_mean,prob_class_std,prob_class_min,prob_class_max,confidence_mean,confidence_min,confidence_max,entropy_mean
,String,String,String,Int64,Int64,Int64,Int64,Float64,Float64,String,Float64,Bool,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,resnet18_pretrained,erp_core_n170_clean,ERP CORE N170,70,35,32,38,0.457143,0.542857,no_class,0.542857,false,0.440003,0.422061,0.00114411,0.999999,0.902298,0.511203,0.999999,0.23549
2,resnet18_pretrained,erp_core_n2pc_clean,ERP CORE N2PC,70,35,2,68,0.0285714,0.971429,no_class,0.971429,false,0.069573,0.13717,4.71993e-5,0.699267,0.939911,0.587162,0.999953,0.161425
3,resnet18_pretrained,eye_eeg_freeviewing_fixations,EYE-EEG FREEVIEWING FIXATIONS,25,25,23,2,0.92,0.08,class,0.92,false,0.892262,0.225853,0.131651,1.0,0.9223,0.507123,1.0,0.142662
4,resnet18_pretrained,eye_eeg_reading_fixations,EYE-EEG READING FIXATIONS,66,66,7,59,0.106061,0.893939,no_class,0.893939,false,0.116769,0.2246,5.79958e-5,0.980798,0.923775,0.509311,0.999942,0.175358
5,resnet18_pretrained,eye_eeg_sceneviewing_tobii_fixations,EYE-EEG SCENEVIEWING TOBII FIXATIONS,3,3,0,3,0.0,1.0,no_class,1.0,true,0.156192,0.267037,5.48401e-5,0.464531,0.843808,0.535469,0.999945,0.23908
6,resnet18_pretrained,roamm_reading_fixations,ROAMM READING FIXATIONS,64,64,11,53,0.171875,0.828125,no_class,0.828125,false,0.192616,0.341627,2.95267e-7,0.999996,0.947741,0.518755,1.0,0.145486
7,resnet34_pretrained,erp_core_n170_clean,ERP CORE N170,70,35,36,34,0.514286,0.485714,class,0.514286,false,0.505941,0.384459,0.000137751,0.998446,0.849725,0.509645,0.999862,0.330586
8,resnet34_pretrained,erp_core_n2pc_clean,ERP CORE N2PC,70,35,21,49,0.3,0.7,no_class,0.7,false,0.303838,0.345878,0.000104286,0.99071,0.863405,0.505538,0.999896,0.295481
9,resnet34_pretrained,eye_eeg_freeviewing_fixations,EYE-EEG FREEVIEWING FIXATIONS,25,25,23,2,0.92,0.08,class,0.92,false,0.851468,0.244955,0.0876671,0.999976,0.90491,0.500321,0.999976,0.226844


## Mod-Split Consistency

For the ERP CORE target data, this table checks whether the two modulo-2 variants from the same origin image receive the same predicted class. A low same-prediction rate would indicate that the model is sensitive to the exact trial subset rather than only to the originating ERP image.

In [5]:
result.consistency_summary_df

Row,model_name,dataset_key,dataset_label,n_modsplit_origins,n_same_prediction,n_disagree_prediction,same_prediction_rate,mean_prob_class_range,max_prob_class_range,mean_confidence_range
,String,String,String,Int64,Int64,Int64,Float64,Float64,Float64,Float64
1,resnet18_pretrained,erp_core_n170_clean,ERP CORE N170,35,25,10,0.714286,0.23492,0.972191,0.11057
2,resnet18_pretrained,erp_core_n2pc_clean,ERP CORE N2PC,35,33,2,0.942857,0.0971512,0.628616,0.0781834
3,resnet34_pretrained,erp_core_n170_clean,ERP CORE N170,35,25,10,0.714286,0.230608,0.720588,0.149787
4,resnet34_pretrained,erp_core_n2pc_clean,ERP CORE N2PC,35,32,3,0.914286,0.171776,0.765679,0.142812


## Training Fit and Output Weights

The fit metrics below are measured on the full labeled fixation training set after final training. The full output-head weights are written to `model_output_head_weights.csv`, and the per-image target logits/probabilities are written to `target_predictions.csv`.

In [6]:
result.train_metrics_df

Row,model_name,depth,nepochs,lr,batchsize,train_time_s,pretrained_params_loaded,accuracy,balanced_accuracy,macro_f1,precision,recall,prob_class_mean,confidence_mean
,String,Int64,Int64,Float32,Int64,Float64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,resnet18_pretrained,18,8,0.0003,32,43.3301,62,0.996894,0.997788,0.996301,0.994845,0.997788,0.301401,0.994109
2,resnet34_pretrained,34,8,0.0003,32,19.3315,110,0.998447,0.998894,0.998148,0.997409,0.998894,0.300277,0.998547


## Figures

Each model receives one prediction-distribution plot and one confidence-example plot per target dataset. The example plots contain up to four unique-origin high-confidence `class` predictions and up to four unique-origin high-confidence `no_class` predictions.

In [7]:
result.figure_df

Row,file
,String
1,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet18_pretrained_prediction_distribution.png
2,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet18_pretrained__erp_core_n2pc_clean__confidence_examples.png
3,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet18_pretrained__erp_core_n170_clean__confidence_examples.png
4,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet18_pretrained__eye_eeg_freeviewing_fixations__confidence_examples.png
5,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet18_pretrained__eye_eeg_reading_fixations__confidence_examples.png
6,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet18_pretrained__eye_eeg_sceneviewing_tobii_fixations__confidence_examples.png
7,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet18_pretrained__roamm_reading_fixations__confidence_examples.png
8,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet34_pretrained_prediction_distribution.png
9,/home/benjamin/Dokumente/BA2/notebooks/week_20/outputs/resnet_fixation_generalization/resnet34_pretrained__erp_core_n2pc_clean__confidence_examples.png


## Interpretation Notes

Use `target_prediction_summary.csv` to inspect whether a model collapses into a single dominant output class for a data source. Use `target_predictions.csv` for the raw logits, probabilities, confidence, and entropy per image. Use the mod-split consistency CSVs to inspect whether the N2PC/N170 modulo variants from the same origin image are classified consistently.